In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
from astropy.coordinates import SkyCoord
import astropy.units as u
from researchcodes import (
    define_column_desc, 
    iter_multi_csv_chunks, 
    write_std_h5, 
)

# Process the text catalog

The RA and Dec errors need calculation, so it's better to process ahead of reading and writing.

In the meantime, we can drop unnecessary columns.

In [3]:
all_column_names = [
    "id_name",
    "chart",
    "ra",
    "dec",
    "V", "B-V", "U-B", "V-R", "R-I", "V-I",
    "n", "m",
    "e_V", "e_B-V", "e_U-B", "e_V-R", "e_R-I", "e_V-I",
]

raw_text_catalog = pd.read_csv(
    "ESO Landolt Equatorial Standards.csv",
    sep=",",
    header=0,
    names=all_column_names,
    comment="#",      # 直接跳过所有 ### 开头的说明行
)

In [4]:
raw_text_catalog

,id_name,chart,ra,dec,V,B-V,U-B,V-R,R-I,V-I,n,m,e_V,e_B-V,e_U-B,e_V-R,e_R-I,e_V-I
0,TPHE A,TPHE.gif,00 30 09,-46 31 22,14.65,0.793,0.380,0.435,0.405,0.841,29,12,0.0028,0.0046,0.0071,0.0019,0.0035,0.0032
1,TPHE B,TPHE.gif,00 30 16,-46 27 55,12.33,0.405,0.156,0.262,0.271,0.535,29,17,0.0115,0.0026,0.0039,0.0020,0.0019,0.0035
2,TPHE C,TPHE.gif,00 30 17,-46 32 34,14.38,-0.298,-1.217,-0.148,-0.211,-0.360,39,23,0.0022,0.0024,0.0043,0.0038,0.0133,0.0149
3,TPHE D,TPHE.gif,00 30 18,-46 31 11,13.12,1.551,1.871,0.849,0.810,1.663,37,23,0.0033,0.0030,0.0118,0.0015,0.0023,0.0030
4,TPHE E,TPHE.gif,00 30 19,-46 24 36,11.63,0.443,-0.103,0.276,0.283,0.564,34,8,0.0017,0.0012,0.0024,0.0007,0.0015,0.0019
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
521,115 268,SA115.gif,23 42 30,+00 52 09,12.49,0.634,0.077,0.366,0.348,0.714,1,1,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
522,115 420,SA115.gif,23 42 36,+01 05 56,11.16,0.468,-0.027,0.286,0.293,0.580,51,42,0.0013,0.0011,0.0020,0.0013,0.0010,0.0015
523,115 271,SA115.gif,23 42 41,+00 45 08,9.70,0.615,0.101,0.353,0.349,0.701,77,55,0.0010,0.0010,0.0014,0.0009,0.0008,0.0011
524,115 516,SA115.gif,23 44 15,+01 14 11,10.43,1.028,0.759,0.563,0.534,1.098,60,49,0.0012,0.0010,0.0019,0.0009,0.0008,0.0010


In [5]:
# calculate the U, B, I, I magnitudes and errors

raw_text_catalog["B"] = raw_text_catalog["B-V"] + raw_text_catalog["V"]

raw_text_catalog["U"] = raw_text_catalog["U-B"] + raw_text_catalog["B"]

raw_text_catalog["R"] = raw_text_catalog["V"] - raw_text_catalog["V-R"]

raw_text_catalog["I"] = raw_text_catalog["V"] - raw_text_catalog["V-I"]

raw_text_catalog["e_B"] = np.sqrt(
    raw_text_catalog["e_B-V"]**2 + raw_text_catalog["e_V"]**2
)

raw_text_catalog["e_U"] = np.sqrt(
    raw_text_catalog["e_U-B"]**2 + raw_text_catalog["e_B"]**2
)

raw_text_catalog["e_R"] = np.sqrt(
    raw_text_catalog["e_V-R"]**2 + raw_text_catalog["e_V"]**2
)

raw_text_catalog["e_I"] = np.sqrt(
    raw_text_catalog["e_V-I"]**2 + raw_text_catalog["e_V"]**2
)

In [6]:
raw_text_catalog

,id_name,chart,ra,dec,V,B-V,U-B,V-R,R-I,V-I,...,e_R-I,e_V-I,B,U,R,I,e_B,e_U,e_R,e_I
0,TPHE A,TPHE.gif,00 30 09,-46 31 22,14.65,0.793,0.380,0.435,0.405,0.841,...,0.0035,0.0032,15.443,15.823,14.215,13.809,0.005385,0.008911,0.003384,0.004252
1,TPHE B,TPHE.gif,00 30 16,-46 27 55,12.33,0.405,0.156,0.262,0.271,0.535,...,0.0019,0.0035,12.735,12.891,12.068,11.795,0.011790,0.012419,0.011673,0.012021
2,TPHE C,TPHE.gif,00 30 17,-46 32 34,14.38,-0.298,-1.217,-0.148,-0.211,-0.360,...,0.0133,0.0149,14.082,12.865,14.528,14.740,0.003256,0.005394,0.004391,0.015062
3,TPHE D,TPHE.gif,00 30 18,-46 31 11,13.12,1.551,1.871,0.849,0.810,1.663,...,0.0023,0.0030,14.671,16.542,12.271,11.457,0.004460,0.012615,0.003625,0.004460
4,TPHE E,TPHE.gif,00 30 19,-46 24 36,11.63,0.443,-0.103,0.276,0.283,0.564,...,0.0015,0.0019,12.073,11.970,11.354,11.066,0.002081,0.003176,0.001838,0.002550
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
521,115 268,SA115.gif,23 42 30,+00 52 09,12.49,0.634,0.077,0.366,0.348,0.714,...,0.0000,0.0000,13.124,13.201,12.124,11.776,0.000000,0.000000,0.000000,0.000000
522,115 420,SA115.gif,23 42 36,+01 05 56,11.16,0.468,-0.027,0.286,0.293,0.580,...,0.0010,0.0015,11.628,11.601,10.874,10.580,0.001703,0.002627,0.001838,0.001985
523,115 271,SA115.gif,23 42 41,+00 45 08,9.70,0.615,0.101,0.353,0.349,0.701,...,0.0008,0.0011,10.315,10.416,9.347,8.999,0.001414,0.001990,0.001345,0.001487
524,115 516,SA115.gif,23 44 15,+01 14 11,10.43,1.028,0.759,0.563,0.534,1.098,...,0.0008,0.0010,11.458,12.217,9.867,9.332,0.001562,0.002460,0.001500,0.001562


In [7]:
# drop the columns we don't want!

process_text_catalog = raw_text_catalog.drop(
    columns=[
        "chart",
        "B-V", "U-B", "V-R", "R-I", "V-I",
        "n", "m",
        "e_B-V", "e_U-B", "e_V-R", "e_R-I", "e_V-I",

    ]
)
                      

In [8]:
# convert the ra and dec to degrees
coords = SkyCoord(
    ra = process_text_catalog["ra"].to_list(),
    dec = process_text_catalog["dec"].to_list(),
    unit = (u.hourangle, u.deg),
    frame = "icrs",
)

process_text_catalog["ra"] = coords.ra.deg
process_text_catalog["dec"] = coords.dec.deg

In [9]:
process_text_catalog

,id_name,ra,dec,V,e_V,B,U,R,I,e_B,e_U,e_R,e_I
0,TPHE A,7.537500,-46.522778,14.65,0.0028,15.443,15.823,14.215,13.809,0.005385,0.008911,0.003384,0.004252
1,TPHE B,7.566667,-46.465278,12.33,0.0115,12.735,12.891,12.068,11.795,0.011790,0.012419,0.011673,0.012021
2,TPHE C,7.570833,-46.542778,14.38,0.0022,14.082,12.865,14.528,14.740,0.003256,0.005394,0.004391,0.015062
3,TPHE D,7.575000,-46.519722,13.12,0.0033,14.671,16.542,12.271,11.457,0.004460,0.012615,0.003625,0.004460
4,TPHE E,7.579167,-46.410000,11.63,0.0017,12.073,11.970,11.354,11.066,0.002081,0.003176,0.001838,0.002550
...,...,...,...,...,...,...,...,...,...,...,...,...,...
521,115 268,355.625000,0.869167,12.49,0.0000,13.124,13.201,12.124,11.776,0.000000,0.000000,0.000000,0.000000
522,115 420,355.650000,1.098889,11.16,0.0013,11.628,11.601,10.874,10.580,0.001703,0.002627,0.001838,0.001985
523,115 271,355.670833,0.752222,9.70,0.0010,10.315,10.416,9.347,8.999,0.001414,0.001990,0.001345,0.001487
524,115 516,356.062500,1.236389,10.43,0.0012,11.458,12.217,9.867,9.332,0.001562,0.002460,0.001500,0.001562


In [10]:
# save the process catalog text file

process_text_catalog.to_csv(
    "ESO_Landolt_Equatorial_Standards.dat",
    sep=",",
    index=False,
)

# Define the HFD5 column description

In [11]:
# Define magnitude column names

# Note this magnitude column order is not necessarily
# to be the same as the column order in the text file. 

# It is OK as long as the filter names in 
# `magnitude_column_names` matches `colnames` defined 
# when reading the text file

magnitude_column_names = [
     "Bessel_U","Bessel_B","Bessel_V","Bessel_R","Bessel_I",
    "Bessel_U_err","Bessel_B_err","Bessel_V_err","Bessel_R_err","Bessel_I_err",
]

# define h5 file column description
h5_columns = define_column_desc(
    magnitude_column_names=magnitude_column_names, 
    id_name_length=20, 
)

In [12]:
h5_columns

{'id_name': StringCol(itemsize=20, shape=(), dflt=np.bytes_(b''), pos=0),
 'ra': Float32Col(shape=(), dflt=np.float32(0.0), pos=1),
 'ra_err': Float32Col(shape=(), dflt=np.float32(0.0), pos=2),
 'dec': Float32Col(shape=(), dflt=np.float32(0.0), pos=3),
 'dec_err': Float32Col(shape=(), dflt=np.float32(0.0), pos=4),
 'Bessel_U': Float32Col(shape=(), dflt=np.float32(0.0), pos=5),
 'Bessel_B': Float32Col(shape=(), dflt=np.float32(0.0), pos=6),
 'Bessel_V': Float32Col(shape=(), dflt=np.float32(0.0), pos=7),
 'Bessel_R': Float32Col(shape=(), dflt=np.float32(0.0), pos=8),
 'Bessel_I': Float32Col(shape=(), dflt=np.float32(0.0), pos=9),
 'Bessel_U_err': Float32Col(shape=(), dflt=np.float32(0.0), pos=10),
 'Bessel_B_err': Float32Col(shape=(), dflt=np.float32(0.0), pos=11),
 'Bessel_V_err': Float32Col(shape=(), dflt=np.float32(0.0), pos=12),
 'Bessel_R_err': Float32Col(shape=(), dflt=np.float32(0.0), pos=13),
 'Bessel_I_err': Float32Col(shape=(), dflt=np.float32(0.0), pos=14),
 'ipix': Int32Col(s

# Read the cvs files

In [13]:
# file path
root_dir = Path(".")
file = root_dir / "ESO_Landolt_Equatorial_Standards.dat"

In [14]:
# csv column names
# here the columns names must match the h5_columns
colnames = [
    "id_name",
    "ra", "dec",
    "Bessel_V",
    "Bessel_V_err",
    "Bessel_B",
    "Bessel_U",
    "Bessel_R",
    "Bessel_I",
    "Bessel_B_err",
    "Bessel_U_err",
    "Bessel_R_err",
    "Bessel_I_err",
]

dataframe_iterator = iter_multi_csv_chunks(
    files=file, 
    chunksize=100, 
    read_csv_kwargs={
        "sep": ",", 
        "engine": "python", 
        "header": None, 
        "names": colnames,
        "skiprows": 1, 
    }
)

In [15]:
table_attrs = {
    "source": "https://www.eso.org/sci/observing/tools/standards/Landolt.html",
    "ra_unit": "deg",
    "dec_unit": "deg",
    "ra_err_unit": "arcsec",
    "dec_err_unit": "arcsec",
    "version": "Landolt 1992",
    "mag_system": {
        "Bessel_U": "Johnson–Kron–Cousins",
        "Bessel_B": "Johnson–Kron–Cousins",
        "Bessel_V": "Johnson–Kron–Cousins",
        "Bessel_R": "Johnson–Kron–Cousins",
        "Bessel_I": "Johnson–Kron–Cousins",
    },
}

In [16]:
write_std_h5(
    dataframe_iterator=dataframe_iterator, 
    ra_dec_hmsdms=False,
    h5_output_path=Path("ESO_Landolt_Equatorial_Standards.h5",), 
    group_where="/ESO", 
    group_name="landolt", 
    group_title="Landolt Equatorial Standards", 
    table_name="std", 
    table_description=h5_columns, 
    table_title="Standard Stars", 
    table_attrs=table_attrs, 
    nside=512, 
    bucket_size=1536 , 
)

Files:   0%|          | 0/1 [00:00<?, ?it/s]

Chunks:   0%|                                                                                                 …